# LangChain 기초

## LangChain 이란?

LLM(대형 언어 모델, Large Language Model)은 수많은 텍스트 데이터를 학습하여 사람처럼 언어를 이해하고 생성할 수 있도록 만들어진 인공지능(AI) 모델이다.  
LLM을 활용하여 애프리케이션과 파이프라인을 신속하게 구축할 수 있는 프레임워크 이다.  
챗봇, 질의응답(QnA), 요약 등 다양한 용도로 활용한다.

## 주요 특징

LLM 애플리케이션 개발에 필요한 다양 구성 요소를 '연결(chain)'하는 데 중점을 둔다.  
다양한 LLM 모델, 프롬프트 템플릿, 검색기(Retriever) 등을 제공한다.  
사용자는 다양한 요소를 '연결(chain)'하는 방식으로 편리하게 시스템 개발 및 구현이 가능하다.

<img src="./langchain.png" width="700" align="left" />

# 환경 구성

## 라이브러리 설치

파이썬의 패키지 관리자인 pip를 사용하여 LLM 기반 애플리케이션 개발에 필요한 라이브러리를 설치한다.  
`pip install -q 설치할라이브러리`에서 `-q` 옵션은 설치 과정을 아주 간결하게 보여주거나 생략하라는 옵션이다. 화면이 지저분해지는 것을 막아준다.

`langchain`  
LLM을 활용한 앱을 만들 때 사용하는 라이브러리로 프롬프트 관리, 데이터 연결, 여러 단계를 묶는 '체인(chain)' 생성 등 복잡한 작업을 쉽게 만들어 준다.  
`langchain-openai`  
langchain과 openai를 연결해주는 라이브러리로 openai의 모델을 langchain 환경에서 호출할 수 있게 한다.  
`tiktoken`  
openai에서 개발한 BPE(Byte Pair Encoding) 토큰 생성 라이브러리로 텍스트가 몇 개의 토큰으로 구성되는지 계산하며, 모델의 토큰 제한 확인이나 비용 예측시 사용된다.

In [1]:
# !pip install langchain langchain-openai tiktoken

## openai 인증키 설정

파이썬 프로그램을 실행하는 환경에서 openai API 서비스를 이용하기 위한 인증 키를 등록한다.

`os`
운영체제(Operation System, OS)와 상호작용하기 위한 파이썬 내장 라이브러리이다.  
파일의 경로 확인, 폴더 생성, `시스템의 환경 변수`를 제어할 때 사용한다.

In [2]:
import os

환경 변수 설정  
컴퓨터나 메모리에 'OPENAI_API_KEY'라는 이름의 보관함을 만들고, 그 안에 openai에서 발급받은 API Key 문자열을 저장한다.  
이후 langchain이나 openai 라이브러리를 사용할 때, 사용자가 매번 API Key를 입력하지 않아도 라이브러리가 이 환경 변수값을 자동으로 찾아가서 인증한다.

In [3]:
# os.environ['OPENAI_API_KEY'] = '사용자 API Key'


# LLM chain

## prompt + LLM

가장 기본적이고 일반적인 사용 사례로 프롬프트 템플릿과 모델을 연결한다.

<img src="./llmchain.png" width="800" align="left" />

과금 정책: https://developers.openai.com/api/docs/pricing

langchain 프레임워크를 사용해서 openai의 LLM에게 질문을 던지고 응답을 받는 가장 기초적이고 핵심적인 구조이다.

openai를 사용하기 위해서 langchain_openai 라이브러리에서 openai의 채팅 전용 모델을 제어하는 클래스인 ChatOpenAI를 사용하기 위해 import 한다.

In [4]:
# 패키지 불러오기
# ChatOpenAI는 우리가 보낸 텍스트를 openai 서버가 이해할 수 있는 형식으로 바꾸고 응답을 받아오는 역할을 한다.
from langchain_openai import ChatOpenAI

# 모델 객체 생성
# 사용할 인공지능 모델의 종류를 지정한다.
# gpt-4o-mini: openai 경량화 멀티모달 ai 모델로, 속도가 빠르고 비용이 저렴하면서도 성능이 뛰어난 가성비 모델이다.
llm = ChatOpenAI(model='gpt-4o-mini')

# 실행(호출)
# invoke() 메소드는 특정 작업(질문)을 수행하며 인수로 지정된 문자열을 모델에게 보내고 모델이 생성한 답변 데이터(객체)를 얻어온다.
result = llm.invoke('지구의 자전 주기는?')

In [5]:
result.content

'지구의 자전 주기는 약 24시간입니다. 정확히 말하면, 태양에 대해 한 바퀴 도는 시간(태양일)은 약 24시간(23시간 56분 4초)입니다. 그러나 우리가 일반적으로 사용하는 하루(24시간)는 지구가 태양 주위를 도는 공전 때문에 약간의 차이가 있습니다. 이 시간을 기준으로 하루를 나누어 24시간으로 사용하고 있습니다.'

랭체인의 강력한 특징인 LCEL(LangChain Expression Language)을 사용해서 `프롬프트 구성 => 모델 호출 => 출력 정제`로 이어지는 과정을 `파이프라인`으로 연결한다.

사용자의 입력을 특정 양식(페르소나 부여 등)에 넣어서 모델에게 전달할 프롬프트 템플릿을 만들기 위해 import 한다.  
`페르소나(persona)`는 LLM에 부여하는 `역할, 성격, 정체성, 말투, 전문 분야 등의 설정`을 의미한다. 페르소나는 라틴어로 '탈' 또는 '가면'을 뜻하는데, AI 분야에서는 `모델에게 특정한 인격이나 역할을 연기하도록 지시하는 프롬프트 기법`으로 사용됀다.

In [6]:
# 사용자 입력을 특정 양식에 넣어서 프롬프를 만드는 역할을 한다.
from langchain_core.prompts import ChatPromptTemplate

모델의 복잡한 응답 객체에서 텍스트 내용만 가져오기 위해 import 한다.

In [7]:
# 모델의 응답 객체에서 텍스트만 가져오는 역할을 한다.
from langchain_core.output_parsers import StrOutputParser

In [8]:
# 프롬프트 정의
# 모델에게 역할인 '당신은 천문학 전문가입니다.'라는 정체성을 부여한다.
# from_template() 메소드의 인수로 프롬프트로 구성할 내용을 넘겨서 프롬프트를 만든다.
# {input} 부분은 나중에 사용자가 입력하는 질문이 들어갈 변수 자리이다.
prompt = ChatPromptTemplate.from_template('당신은 천문학 전문가입니다. 다음 질문에 답변해 주세요: {input}')

# 모델 설정
llm = ChatOpenAI(model='gpt-4o-mini')

# 파서 설정
# 모델이 리턴하는 메타 데이터는 버리고, 우리가 읽을 수 있는 문자열만 골라내도록 설정한다.
output_parser = StrOutputParser()

# LCEL chaining
# 파이프 기호(|)를 사용해서 prompt, llm, output_parser를 하나로 묶는다. 앞의 출력이 자동으로 뒤의 입력으로 들어간다.
# 사용자 입력이 prompt에 전달되서 prompt를 만들고 작성된 prompt가 llm으로 전달되고 인공지능이 응답한 결과가 output_parser로 전달된다.
chain = prompt | llm | output_parser

# 프롬프트와 모델을 파이프라인으로 연결한 경우 체인을 실행하기 전에 프롬프트 단계만 따로 실행해 볼 수 있다.
# print(prompt.invoke({'input': '지구의 자전 주기는?'}))

# chain 실행
# invoke() 메소드로 전체 LCEL chain을 실행한다.
# 템플릿에 정의한 {input} 변수에 실제 질문인 '지구의 자전 주기는?'을 딕셔너리 형태로 전달한다.
result = chain.invoke({'input': '지구의 자전 주기는?'})

In [9]:
result

'지구의 자전 주기는 약 24시간입니다. 이 시간은 지구가 한 번 자전을 하여 태양에 대해 같은 지점을 다시 향할 때까지 걸리는 시간을 기준으로 합니다. 정확하게는 평균적으로 24시간 0분 4초 정도입니다. 이 시간을 기준으로 하루가 정의됩니다.'

## multiple chain

In [10]:
# 프롬프트 정의
prompt1 = ChatPromptTemplate.from_template('{korean_word}을(를) 영어로 번역해 줘.')

# 모델 설정
llm = ChatOpenAI(model='gpt-4o-mini')

# 체인 생성
# LCEL(파이프라인) 문법을 사용해서 3개를 하나로 묶어준다.
# prompt1(번역 지시) => llm(모델 호출 및 응답) => StrOutputParser()(응답 결과에서 문자열만 추출)순으로 동작하는 chain1을 정의한다.
chain1 = (prompt1 | llm | StrOutputParser())

# 체인 실행
# invoke() 메소드가 실행되면 '미래'라는 문자열을 프롬프트의 {korean_word}에 전달한다.
chain1.invoke({'korean_word': '미래'})

'"미래"는 영어로 "future"입니다.'

In [11]:
# 프롬프트 정의
prompt2 = ChatPromptTemplate.from_template('{english_word}의 뜻을 옥스퍼드 사전을 바탕으로 한글로 설명해 줘.')

# 체인 생성
chain2 = (
    # prompt2는 {english_word}라는 변수에 값을 입력받아야 한다.
    # 먼저 chain1을 실행하고 얻은 결과값('future')을 {english_word}라는 변수에 전달한다.
    {'english_word': chain1}
    # chain1의 실행 결과가 {english_word}에 전달된 내용을 바탕으로 프롬프트를 만든다.
    | prompt2
    # 완성된 프롬프트가 gpt-4o-mini 모델로 전달되고 모델은 옥스퍼드 사전을 참고해서 답변을 생성한다.
    | llm
    # 모델이 리턴한 응답 데이터에서 불필요한 정보는 제외하고 문자열만 뽑아낸다.
    | StrOutputParser()
)

# 체인 실행
chain2.invoke({'korean_word': '미래'})

'"미래"라는 단어는 옥스퍼드 사전을 바탕으로 설명하자면, 특정한 시점 이후에 발생할 수 있는 사건이나 상황을 나타내는 개념입니다. 이는 현재와 과거와 구분되며, 시간적으로 아직 도달하지 않은 시점을 의미합니다. 미래는 종종 예측이나 계획, 희망과 관련되어 언급됩니다. 예를 들어, 사람들은 미래에 대한 기대나 불안감을 느끼기도 하며, 다양한 가능성이 존재하는 시점으로 이해할 수 있습니다.'